# ③ 검증 — 평가 데이터셋 만들기
담당: 검증 · 코드: `rag/validation/validation.py`

1. 01 전처리의 `artifacts/chunks.jsonl` 로 QA(질문 · 정답 · 근거 키워드)를 만들고, 근거 청크에 실제로 있는 키워드만 남깁니다.
2. QA 를 train / valid / test 로 나눕니다 (평가에는 test 만 사용).
3. 골든 셋(`eval/golden_set.jsonl`) · 평가 질문(`eval/eval_questions.yaml`) · test QA 를 점검해 평가에 쓸 수 있는 항목만 `artifacts/eval_set.jsonl` 로 저장합니다 → 04 생성이 답하고 05 평가가 채점합니다.

QA 생성은 OpenAI API 를 씁니다 (호출 수 = 문서 수 × `qa_per_doc`).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl, write_jsonl
from rag.validation.validation import (check_eval_items, filter_qa, generate_qa, load_eval_items, qa_eval_items,
                                       split_qa)

cfg = load_config()
vcfg = cfg["validation"]

## 1. QA 생성 · 근거 키워드 정리

In [ ]:
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
qa_raw = generate_qa(chunks, vcfg, cfg["llm"]["model"])
qa = filter_qa(qa_raw, vcfg)
write_jsonl(qa, ARTIFACTS / "qa.jsonl")
print(f"생성 {len(qa_raw)}개 → 근거 키워드 정리 후 {len(qa)}개")
pd.DataFrame(qa)[["id", "question", "reference", "evidence_keywords"]].head(10)

## 2. train / valid / test 분할

In [ ]:
splits = split_qa(qa, vcfg)
pd.Series({"train": len(splits["train"]), "valid": len(splits["valid"]), "test": len(splits["test"])}, name="QA 수")

## 3. 평가 항목 점검 · 저장
제외되는 항목은 사유가 `점검` 열에 나옵니다.

In [ ]:
items = load_eval_items(vcfg["golden_sets"]) + qa_eval_items(splits["test"], vcfg["max_qa"])
eval_set, report = check_eval_items(items)
write_jsonl(eval_set, ARTIFACTS / "eval_set.jsonl")
print(f"평가 데이터셋 {len(eval_set)}개 / 점검 {len(items)}개")
report